# 02 · Serie mensual por segmento

**Fase CRISP-DM:** preparación de los datos.

**Objetivo:** construir la serie mensual de toneladas por segmento con las mismas reglas que la curación (`procesamiento/`) y dejarla en un lakehouse de exploración. Es un puente mientras no exista el adaptador CSV de `ingesta/`: los notebooks 03 a 06 leen del lakehouse y no cambian cuando la serie venga de la ingesta real.

**Reglas aplicadas:** solo líneas con kg de GLP; las notas de crédito restan; la venta de envases se excluye; el segmento sale de `referencias/mapa_segmentos.csv`; la serie corta en el último mes completo.

**Regla de privacidad:** no se muestran filas. Se leen solo las columnas necesarias y `cliente_id` se usa únicamente para contar clientes distintos por mes y segmento.

In [ ]:
import os
from pathlib import Path

import exp_a
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
from IPython.display import display

from lojagas.comun import lakehouse
from lojagas.procesamiento.agregacion import (
    EXCLUIR,
    a_mensual,
    asignar_segmento,
    cargar_mapa_segmentos,
)
from lojagas.procesamiento.contratos import ventas_mensual_curated
from lojagas.procesamiento.curacion import hasta_periodo

ENTRADA = Path(os.environ["LOJAGAS_DATOS_ENTRADA"]).expanduser()
VENTAS = ENTRADA / "ventas_2026-11-13.csv"
PERIODO = "2026-09-01"  # último mes completo: la exportación corta a mitad de octubre
MAPA = Path(exp_a.__file__).parents[1] / "referencias" / "mapa_segmentos.csv"
RAIZ = exp_a.raiz_lakehouse()
assert VENTAS.exists() and MAPA.exists()
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
exp_a.estilo_graficos()

## Lectura

Los errores de lectura de pyarrow incluyen el texto de la fila. Por eso las filas malformadas se cuentan y se saltan sin guardar su contenido, y cualquier otro error se relanza sin detalle.

In [ ]:
COLUMNAS = [
    "factura_id",
    "linea_id",
    "fecha_factura",
    "move_type",
    "state",
    "categoria",
    "kg_glp",
    "cliente_id",
]
TIPOS = {
    "factura_id": pa.int64(),
    "linea_id": pa.int64(),
    "fecha_factura": pa.date32(),
    "kg_glp": pa.float64(),
}


def leer(ruta, columnas):
    saltadas = 0

    def saltar(_fila):
        # No se guarda ni se imprime _fila.text: trae todas las columnas del CSV.
        nonlocal saltadas
        saltadas += 1
        return "skip"

    try:
        tabla = pv.read_csv(
            ruta,
            parse_options=pv.ParseOptions(newlines_in_values=True, invalid_row_handler=saltar),
            convert_options=pv.ConvertOptions(
                include_columns=columnas,
                strings_can_be_null=True,
                null_values=["NULL", ""],
                column_types=TIPOS,
            ),
        )
    except pa.ArrowException as e:
        # El mensaje original puede incluir valores del CSV; solo se informa el tipo de error.
        raise RuntimeError(
            f"No se pudo leer {ruta.name}: {type(e).__name__} (detalle suprimido)"
        ) from None
    print(f"filas malformadas saltadas: {saltadas}")
    return tabla.to_pandas(date_as_object=False)


lineas = leer(VENTAS, COLUMNAS)
print(f"líneas leídas: {len(lineas)}")

## De líneas de factura a serie mensual

Primero, las mismas comprobaciones que exige el contrato de Raw. Después, las funciones de `procesamiento/`, sin copiar su lógica.

In [ ]:
assert lineas["linea_id"].is_unique, "linea_id duplicado: deduplicar antes de sumar"
assert lineas["move_type"].isin(["out_invoice", "out_refund"]).all(), "move_type inesperado"
assert (lineas["state"] == "posted").all(), "hay documentos sin publicar"

lineas = hasta_periodo(lineas, PERIODO)
mapa = cargar_mapa_segmentos(MAPA)
serie = ventas_mensual_curated.validate(a_mensual(lineas, mapa))
print(
    f"serie: {serie['mes'].nunique()} meses x {serie['segmento'].nunique()} segmentos, "
    f"de {serie['mes'].min():%Y-%m} a {serie['mes'].max():%Y-%m}"
)

In [ ]:
# Comprobación independiente: el mismo total calculado sin pasar por el mapa de segmentos
glp = lineas[lineas["categoria"].str.startswith("GLP /", na=False)]
signo = glp["move_type"].map({"out_invoice": 1.0, "out_refund": -1.0})
esperado = (signo * glp["kg_glp"]).sum() / 1000
total = serie["toneladas"].sum()
print(f"toneladas en la serie: {total:,.1f} | calculadas aparte: {esperado:,.1f}")
assert abs(total - esperado) < 1e-6 * esperado
assert (serie["toneladas"] > 0).all(), "hay meses en cero o negativos: revisar antes de modelar"

## Clientes activos por mes y segmento

Cuántos clientes distintos compraron GLP en el mes. Es un conteo: ningún identificador sale de esta celda.

In [ ]:
con_kg = lineas[(lineas["kg_glp"].fillna(0) > 0) & (lineas["move_type"] == "out_invoice")].copy()
con_kg["segmento"] = asignar_segmento(con_kg, mapa)
con_kg = con_kg[con_kg["segmento"] != EXCLUIR]
con_kg["mes"] = (
    pd.to_datetime(con_kg["fecha_factura"])
    .dt.to_period("M")
    .dt.to_timestamp()
    .astype("datetime64[ns]")
)
conteo = con_kg.groupby(["mes", "segmento"])["cliente_id"].nunique().rename("clientes_activos")
clientes = (
    serie[["mes", "segmento"]]
    .merge(conteo.reset_index(), on=["mes", "segmento"], how="left")
    .fillna({"clientes_activos": 0})
    .astype({"clientes_activos": "int64"})
)

## Actividad por día de la semana

Cuánto se vende cada día de la semana frente a un día promedio (media 1). Con esto se cuentan los días de venta de cada mes: un mes con cinco domingos no vende lo mismo que uno con cuatro. Se calcula **solo con el primer año**, anterior a cualquier mes de prueba, para que la variable no use información del futuro.

In [ ]:
ANIO_PESOS = int(serie["mes"].dt.year.min())
base = con_kg[con_kg["fecha_factura"].dt.year == ANIO_PESOS]
rejilla = pd.MultiIndex.from_product(
    [sorted(serie["segmento"].unique()), range(7)], names=["segmento", "dia_semana"]
)
kg_dia = (
    base.assign(dia_semana=base["fecha_factura"].dt.day_of_week)
    .groupby(["segmento", "dia_semana"])["kg_glp"]
    .sum()
    .reindex(rejilla, fill_value=0.0)
)
calendario = pd.date_range(f"{ANIO_PESOS}-01-01", f"{ANIO_PESOS}-12-31").day_of_week
veces = pd.Series(calendario).value_counts()
por_dia = kg_dia / kg_dia.index.get_level_values("dia_semana").map(veces)
pesos = (por_dia / por_dia.groupby(level="segmento").transform("mean")).rename("peso").reset_index()
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
print(f"Actividad relativa por día de la semana en {ANIO_PESOS} (1 = día promedio)")
display(
    pesos.pivot_table(index="segmento", columns="dia_semana", values="peso")
    .set_axis(DIAS, axis=1)
    .round(2)
)
del con_kg, base, lineas

## Escritura en el lakehouse de exploración

Tablas agregadas, sin PII, fuera del repositorio. Cada ejecución crea una versión Delta nueva.

In [ ]:
v_ventas = lakehouse.escribir(serie, RAIZ, "curated", exp_a.TABLA_VENTAS, modo="overwrite")
v_clientes = lakehouse.escribir(clientes, RAIZ, "curated", exp_a.TABLA_CLIENTES, modo="overwrite")
print(f"{exp_a.TABLA_VENTAS}: versión Delta {v_ventas}")
print(f"{exp_a.TABLA_CLIENTES}: versión Delta {v_clientes}")
v_dias = lakehouse.escribir(pesos, RAIZ, "curated", exp_a.TABLA_DIAS, modo="overwrite")
print(f"{exp_a.TABLA_DIAS}: versión Delta {v_dias}")

## Resumen de la serie

In [ ]:
toneladas = serie.pivot_table(index="mes", columns="segmento", values="toneladas")
resumen = toneladas.agg(["mean", "std", "min", "max"]).T
resumen["cv_%"] = resumen["std"] / resumen["mean"] * 100
resumen["participacion_%"] = toneladas.sum() / toneladas.sum().sum() * 100
display(resumen.round(1))

anual = toneladas.groupby(toneladas.index.year).sum()
anual["meses"] = toneladas.groupby(toneladas.index.year).size()
display(anual.round(0))

In [ ]:
segmentos = list(toneladas.columns)
fig, ejes = plt.subplots(len(segmentos), 1, figsize=(10, 2.3 * len(segmentos)), sharex=True)
for eje, segmento in zip(ejes, segmentos, strict=True):
    eje.plot(toneladas.index, toneladas[segmento], color=exp_a.color_segmento(segmento, segmentos))
    eje.set_title(f"{segmento} · toneladas por mes")
    eje.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
display(toneladas.set_axis(toneladas.index.strftime("%Y-%m")).round(1))
activos = clientes.pivot_table(index="mes", columns="segmento", values="clientes_activos")
display(activos.set_axis(activos.index.strftime("%Y-%m")))